# Ranked Carver book — BTC, QQQ, GLD

One book, two layers, one dial.

1. **Relative selection.** Each US session, rank the three names by 60-session ROC. Keep the **top 2** with a positive Carver forecast. That is the QMIE allocator idea: do not size into the laggard.
2. **Absolute sizing.** Carver EWMAC / breakout / accel / skew / cross-sectional momentum → a continuous weight. Then **portfolio vol targeting** scales the whole book so ex-ante vol sits on a dial.
3. **The dial.** Vol target is chosen on **IS only** to sit near **−10% max DD**. Sharpe **1.4–1.5** is a *goal*. We do not search OOS until the band prints.

Research only. QMIE stays signal-only. This notebook does not retune live `W_*` and does not send orders.

| Mandate | Number |
|---|---|
| Max drawdown (hard) | ~10% (accept up to 12% on OOS) |
| Sharpe (goal) | 1.4–1.5 annualized on **252** sessions |
| Fit | IS 2019-09 → 2022-12 (data from first complete trio print) |
| Never-tune | OOS 2023-01 → today |
| Robustness | walk-forward folds + DF neighborhood on inner IS |

Calendar is **US sessions** (QQQ/GLD). BTC is as-of joined from Binance Vision UTC daily — a few hours off 16:00 ET, not interpolated. `exec_lag=1`. Costs 2 bps on turnover.

**Sharpe is not the vol dial.** Scaling the same Carver path changes DD and CAGR; Sharpe stays until a cap or cost bites. The 1.4–1.5 number is a *signal-quality* goal on OOS, not something you print by turning vol up. The 10% DD is the dial.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
import numpy as np
import pandas as pd
from research.trend_lab.carver_book import (
    ANN_SESSIONS, BookParams, TARGET_DD, TARGET_SHARPE,
    book_from_raw_weights, carver_weight_panel, is_oos_index,
    neighborhood, pick_vol_target, slice_kpis, walk_forward,
)
from research.trend_lab.data import mixed_panel
from research.trend_lab.metrics import kpis_from_net, kpi_table
from research.trend_lab.plots import allocation_fig, equity_overlay, rolling_sharpe_fig, underwater
from research.trend_lab.protocol import SPLIT

print("IS", SPLIT.is_start, "→", SPLIT.is_end)
print("OOS", SPLIT.oos_start, "→", SPLIT.oos_end)
print("ann_days", ANN_SESSIONS, "target DD", TARGET_DD, "Sharpe band", TARGET_SHARPE)


## Data

BTC from Binance Vision USDT-M 1d. QQQ and GLD from Stooq (Yahoo fallback), cached under `python/backtest/data/cache/etf/`.


In [ ]:
panel, sources = mixed_panel()
print(sources)
print(panel.describe().T[["mean", "std", "min", "max"]])
corr = panel.pct_change(fill_method=None).corr()
display(corr.round(3))
display(panel.tail(3))
print(len(panel), panel.index[0].date(), "→", panel.index[-1].date())


## Carver forecasts (unit vol) + ranked overlay

Per name: EWMAC, breakout, accel, skew, CS momentum vs the other two. Long-only. Rank keeps top-2 by 60d ROC among names with weight > 0. Then the **book** is scaled to a portfolio vol target using lagged EWM vol of the lagged book — the same master dial as the Carver note, now at portfolio level so three 20% engines do not stack into a 30% book.


In [ ]:
raw_w = carver_weight_panel(panel, use_cs=True, ann_days=ANN_SESSIONS)
print("FDM", raw_w.attrs.get("fdm"))
is_end, oos_start = is_oos_index(panel)
is_p = panel.loc[:is_end]
print("IS bars", len(is_p), "OOS bars", len(panel.loc[oos_start:]))


## IS dial: max DD ≈ 10%, Sharpe 1.4–1.5 if the plateau exists

Grid vol target on IS only. Prefer DD within 2.5 points of 10%. Among those, prefer Sharpe inside 1.4–1.5. If the band is empty, take the closest DD and **do not** chase Sharpe on OOS.


In [ ]:
picked = pick_vol_target(is_p, raw_w.reindex(is_p.index).fillna(0.0), lookback=60, top_n=2)
display(picked["table"][["vol_target", "sharpe", "max_dd", "cagr", "calmar", "dd_gap", "in_sharpe_band"]].round(3))
params = BookParams(vol_target=picked["vol_target"], lookback=60, top_n=2)
print("picked vol_target", params.vol_target)


In [ ]:
book = book_from_raw_weights(panel, raw_w, params)
all3 = book_from_raw_weights(panel, raw_w, BookParams(vol_target=params.vol_target, lookback=60, top_n=3))
brk = book_from_raw_weights(panel, raw_w, BookParams(vol_target=params.vol_target, lookback=60, top_n=2, dd_trip=-0.11))
bh = panel.pct_change(fill_method=None).mean(axis=1).fillna(0.0)

rows = {
    "ranked_IS": slice_kpis(book, is_p.index[0], is_end),
    "ranked_OOS": slice_kpis(book, oos_start, panel.index[-1]),
    "all3_OOS": slice_kpis(all3, oos_start, panel.index[-1]),
    "breaker_OOS": slice_kpis(brk, oos_start, panel.index[-1]),
    "bh_OOS": kpis_from_net(bh.loc[oos_start:], ann=ANN_SESSIONS),
}
tbl = kpi_table(rows)
display(tbl.round(3))
oos = rows["ranked_OOS"]
print("OOS Sharpe in 1.4–1.5:", 1.4 <= oos["sharpe"] <= 1.5)
print("OOS max DD:", round(oos["max_dd"], 4), "(target −0.10, cap −0.12)")


## Robustness — do not skip this

**Walk-forward:** pick the vol dial on each fold's IS, apply to that fold's OOS. If only the 2023–26 bull prints 1.5 Sharpe, the engine is a regime, not a system.

**Neighborhood:** ±20% vol, lookback 40/60/80, top_n 1/2/3, scored on the last 20% of IS. A spike with val Sharpe std ≫ 1 is not a plateau (same lesson as the KAMA DF notebook).


In [ ]:
folds = [
    ("2020-01-01", "2021-12-31", "2022-01-01", "2022-12-31"),
    ("2020-01-01", "2022-12-31", "2023-01-01", "2023-12-31"),
    ("2020-01-01", "2023-12-31", "2024-01-01", "2026-12-31"),
]
wf = walk_forward(panel, raw_w, folds=folds, lookback=60, top_n=2)
display(wf.round(3))
print("WF mean Sharpe", round(float(wf["sharpe"].mean()), 3), "mean DD", round(float(wf["max_dd"].mean()), 3))

nb = neighborhood(is_p, raw_w, params)
display(nb.round(3))
print("inner-val Sharpe std", round(float(nb["sharpe_val"].std(ddof=1)), 3))


## Plots — OOS only for the eye; the numbers above are the test


In [ ]:
oos_idx = panel.loc[oos_start:].index
eq = lambda net: (1.0 + net).cumprod()
equity_overlay({
    "ranked top-2": eq(book.loc[oos_idx]["net"]),
    "all 3": eq(all3.loc[oos_idx]["net"]),
    "DD breaker": eq(brk.loc[oos_idx]["net"]),
    "equal BH": eq(bh.loc[oos_idx]),
}, "OOS growth of $1 — BTC / QQQ / GLD").show()

rolling_sharpe_fig({
    "ranked top-2": book.loc[oos_idx]["net"],
    "all 3": all3.loc[oos_idx]["net"],
    "BH": bh.loc[oos_idx],
}, window=63, title="OOS 63-session rolling Sharpe (ann=252)").show()

underwater(eq(book.loc[oos_idx]["net"]), "Ranked Carver OOS drawdown (target −10%)").show()
allocation_fig({
    "BTC": book.loc[oos_idx]["w_BTC"],
    "QQQ": book.loc[oos_idx]["w_QQQ"],
    "GLD": book.loc[oos_idx]["w_GLD"],
}, "OOS weights — ranked Carver").show()


## How to read a miss

- If OOS Sharpe is 1.1 and DD is −9%, the **DD mandate held**. Forcing vol up on OOS to print 1.45 Sharpe is overfitting. Raise the dial only if IS neighborhood still sits near 10% DD.
- If OOS DD is −18% at the IS-picked dial, the vol model did not transfer (crypto vol regime). That is a REJECT, not a cue to add a second overlay.
- Walk-forward 2022 (rate shock / crypto winter) is the honest fold. A system that only works in 2023–25 is beta, not Carver.
- Three names is a toy book. The point is the *stack*: relative rank → absolute Carver size → portfolio vol dial. QMIE live still does not execute this.
